In [0]:
from mlflow import MlflowClient

In [0]:
#%pip install xgboost

import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load Iris dataset
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create a Databricks MLflow experiment
mlflow.set_experiment("/Users/jordan.bonev@yahoo.com/iris-mlflow-demo")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Dataset shape:", X.shape)
print("Features:", list(X.columns))
print("Species:", list(iris.target_names))

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

In [0]:
mlflow.set_experiment("/Users/jordan.bonev@yahoo.com/iris-mlflow-demo")

In [0]:
models = [
    ("RandomForest", RandomForestClassifier(
        n_estimators=50,
        max_depth=3,
        random_state=42
    )#, ["sepal length (cm)", "petal length (cm)"]
     ),
    ("RandomForest", RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )),
    ("RandomForest", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42
    )),
    ("XGBoost", XGBClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )),
    ("XGBoost", XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    )),
    ("XGBoost", XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    ))
]

In [0]:
for model_name, model in models:

    with mlflow.start_run(run_name=f"{model_name}_{len(models)}"):

        # Identify the algorithm
        mlflow.set_tag("model_family", model_name)

        # Train
        model.fit(X_train, y_train)

        # Predict
        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, average="macro")

        # Log parameters, metrics and model
        mlflow.log_params(model.get_params())
        mlflow.log_param("model_family", model_name)

        mlflow.log_metrics({
            "test_accuracy": accuracy,
            "test_f1_macro": f1
        })

        if model_name == "RandomForest":
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model"
            )
        else:
            mlflow.xgboost.log_model(
                xgb_model=model,
                name="model"
            )

        print(
            f"{model_name}: "
            f"accuracy={accuracy:.3f}, "
            f"macro-F1={f1:.3f}"
        )

In [0]:
experiment = mlflow.get_experiment_by_name(
    "/Users/jordan.bonev@yahoo.com/iris-mlflow-demo"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_f1_macro DESC"]
)

display(
    runs[[
        "tags.model_family",
        "metrics.test_accuracy",
        "metrics.test_f1_macro",
        "run_id"
    ]]
)

In [0]:
best_run_id = runs.iloc[0]["run_id"]

model_uri = f"runs:/{best_run_id}/model"

loaded_model = mlflow.sklearn.load_model(model_uri)

# Predict the species of one flower
sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
probabilities = loaded_model.predict_proba(sample)[0]

print("Predicted species:", iris.target_names[prediction])
print("Probabilities:", probabilities)
print("Actual species:", iris.target_names[y_test.iloc[0]])

In [0]:
# Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

# Registered model name: catalog.schema.model
model_name = "workspace.default.iris_classifier"

# Select the best run based on test macro F1
best_run = runs.iloc[0]
best_run_id = best_run["run_id"]

# Register the model from the winning run
model_uri = f"runs:/{best_run_id}/model"

# UC requires a model signature — re-log the model with one before registering
loaded = mlflow.sklearn.load_model(model_uri)
signature = mlflow.models.infer_signature(X_test, loaded.predict(X_test))

with mlflow.start_run(run_name="uc-registration") as reg_run:
    mlflow.sklearn.log_model(sk_model=loaded, name="model", signature=signature)
    reg_model_uri = f"runs:/{reg_run.info.run_id}/model"

registered_model = mlflow.register_model(
    model_uri=reg_model_uri,
    name=model_name
)

print("Registered model version:", registered_model.version)

# Assign the Champion alias to this version
client.set_registered_model_alias(
    name=model_name,
    alias="Champion",
    version=registered_model.version
)

print("Champion assigned to version:", registered_model.version)

In [0]:
import mlflow.pyfunc

# Load whichever model version currently has the Champion alias
champion_uri = f"models:/{model_name}@Champion"

champion_model = mlflow.pyfunc.load_model(champion_uri)

# Predict on test data
predictions = champion_model.predict(X_test)

print("Predictions:", predictions[:10])